# Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual <a href='https://quantra.quantinsti.com/quantra-notebook' target="_blank">here</a>.
1. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
1. Before running this notebook on your local PC:<br>
i.  You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on "**Run Codes Locally on Your Machine**" in the course.<br>
ii. You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or python modules that might be required to run this notebook.

# Backtest the Head and Shoulders Pattern

In the previous notebook, we were able to identify the presence of multiple head and shoulders patterns within our data. We also stored all the relevant information for each instance of the head and shoulders pattern in a CSV file. In this notebook, let us proceed with backtesting all of these head and shoulders patterns.

This notebook is structured as follows:
1. [Import Libraries](#libraries)
2. [Read the Data](#data)
3. [Merge the Data With Pattern Details](#merge)
4. [Backtest the Strategy](#backtest)
5. [Conclusion and Next Steps](#conclusion)

<a id='libraries'></a>
## Import Libraries

In [1]:
# Import necessary libraries
import pandas as pd

<a id='data'></a>
## Read the Data

Import the file `spy_daily_1993_2018.csv` using the `read_csv` method of `pandas`. This file has the OHLCV values for SPY in the daily frequency. This CSV file is available in the zip file of the unit 'Python Codes and Data' in the 'Course Summary' section.

In [2]:
# Import the stock price data
data = pd.read_csv(
    '../data_modules/spy_daily_1993_2018.csv', index_col=0)

# Change the index type to datetime
data.index = pd.to_datetime(data.index)

# Display the data
data.tail()

,Open,High,Low,Close,Volume
Date,,,,,
2018-12-24,224.666298,226.358066,220.183133,220.248917,147311600
2018-12-26,221.780913,231.376968,219.703796,231.376968,218485400
2018-12-27,227.984041,233.360078,224.591113,233.153320,186267300
2018-12-28,234.572501,236.283055,231.630707,232.852539,153100200
2018-12-31,234.553721,235.145843,232.589398,234.892075,144299400


In the previous notebook, we had created a dataframe by the name `hs_patterns_data` which held details of every single instance of the head and shoulders patterns that was detected for SPY based on the price data. We saved all of this information into a CSV file called `hs_trade_setups.csv`. 

Let us now import the very same data so that we can backtest the head and shoulders pattern.

In [3]:
# Import the pattern details of the inverse head and shoulders
pattern_details = pd.read_csv(
    '../data_modules/hs_trade_setups.csv', index_col=0)

# Convert the confirmation_date column type to datetime
pattern_details.confirmation_date = pd.to_datetime(
    pattern_details.confirmation_date)

# Set the index of pattern_details as 'confirmation_date' column
pattern_details.index = pattern_details.confirmation_date
pattern_details.tail()

,sh1_date,neck1_date,head_date,neck2_date,sh2_date,sh1_price,neck1_price,head_price,neck2_price,sh2_price,confirmation_date,time_for_confirmation,signal,stoploss,head_length,target
confirmation_date,,,,,,,,,,,,,,,,
2006-02-07,2005-12-14,2005-12-30,2006-01-11,2006-01-25,2006-01-30,91.920951,89.717330,93.382150,90.785015,92.927670,2006-02-07,8,-1,93.020598,2.597135,77.799338
2011-09-22,2011-08-17,2011-08-22,2011-08-31,2011-09-12,2011-09-20,97.800056,90.707152,99.664096,92.030513,98.946809,2011-09-22,2,-1,99.045756,7.633583,53.862596
2015-08-20,2015-06-22,2015-07-07,2015-07-20,2015-07-27,2015-07-31,186.217163,178.789133,186.733906,180.672454,185.218543,2015-08-20,20,-1,185.403762,6.061453,150.365190
2016-06-27,2016-04-01,2016-04-07,2016-04-20,2016-05-06,2016-05-11,184.418266,180.812506,187.783649,181.515860,185.664686,2016-06-27,47,-1,185.850351,6.267789,150.176914
2018-10-05,2018-08-29,2018-09-07,2018-09-21,2018-09-26,2018-10-03,271.376965,266.698077,273.988388,270.428348,273.979057,2018-10-05,2,-1,274.253036,3.560040,252.628150


<a id='merge'></a>
## Merge the Data With Pattern Details

We will now merge the two dataframes `data` and `pattern_details` so that all of our data is accessible under a single dataframe called `hs_strategy_data` which we can refer to while backtesting.

In [4]:
# Merge the 'pattern_details' dataframe with the dataframe 'data'
hs_strategy_data = pd.merge(data, pattern_details,
                            left_index=True, right_index=True, how='left')
hs_strategy_data.signal.fillna(0, inplace=True)
hs_strategy_data.tail()

/var/folders/tj/1fwc3zqn73160jbtxl1t558h0000gn/T/ipykernel_26826/2621189036.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  hs_strategy_data.signal.fillna(0, inplace=True)


,Open,High,Low,Close,Volume,sh1_date,neck1_date,head_date,neck2_date,sh2_date,...,neck1_price,head_price,neck2_price,sh2_price,confirmation_date,time_for_confirmation,signal,stoploss,head_length,target
Date,,,,,,,,,,,,,,,,,,,,,
2018-12-24,224.666298,226.358066,220.183133,220.248917,147311600,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,0.0,NaN,NaN,NaN
2018-12-26,221.780913,231.376968,219.703796,231.376968,218485400,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,0.0,NaN,NaN,NaN
2018-12-27,227.984041,233.360078,224.591113,233.153320,186267300,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,0.0,NaN,NaN,NaN
2018-12-28,234.572501,236.283055,231.630707,232.852539,153100200,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,0.0,NaN,NaN,NaN
2018-12-31,234.553721,235.145843,232.589398,234.892075,144299400,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaT,NaN,0.0,NaN,NaN,NaN


<a id='backtest'></a>
## Backtest the Strategy

In the following cells, we will code a backtester function that you can use to backtest all of your price action based trading strategies. 

You can begin by initialising some basic variables as shown under **Initial Settings** in the below code cell.

After initialising these variables, you can proceed to perform the backtest in Python as shown under **Backtest over historical data**. Under this, the very first block of code **Positions Check** will cover the logic for setting the value of the `entry_flag` and the `exit_flag` depending on the state of `current_position` and `signal` values.

The block under **Entry Position Update** highlights the various steps that you need to perform when the `entry_flag` is set to `True` indicating that you need to take a long/short position. 

And finally, in the code under the section **Exit Position Update**, we have specified the various steps that you need to perform when the `exit_flag` is set to `True` indicating that you need to exit your open position. 

Note: We will also save important trade-related details such as `entry_date`, `entry_price`, `exit_date`, `exit_price`, `position`, `exit_type`and `PnL` in a separate dataframe called `round_trips_details`.

In [5]:
def backtester(data):

    # ------------------------------- Initial Settings -------------------------------------
    # Create dataframes for round trips, storing trades, and mtm
    round_trips_details = pd.DataFrame()
    trades = pd.DataFrame()

    # Initialise current position, number of trades, cumulative pnl, stop-loss to 0 and take-profit to 100000
    current_position = 0
    trade_num = 0
    cum_pnl = 0
    sl = 0
    tp = 100000

    # Set exit flag to False
    exit_flag = False
    entry_flag = False

    # ------------------------------- Backtest Over Historical Data -------------------------------------
    for i in data.index:

        # ------------------- Positions Check ----------------------------
        # No positions
        if (current_position == 0) & (data.loc[i, 'signal'] != 0):
            current_position = data.loc[i, 'signal']
            entry_flag = True

        # Short position
        elif current_position == -1:

            if data.loc[i, 'Close'] > stop_loss:
                exit_type = 'SL'
                exit_flag = True

            elif data.loc[i, 'Close'] < take_profit:
                exit_type = 'TP'
                exit_flag = True

        # Long position
        elif current_position == 1:

            if data.loc[i, 'Close'] < stop_loss:
                exit_type = 'SL'
                exit_flag = True

            elif data.loc[i, 'Close'] > take_profit:
                exit_type = 'TP'
                exit_flag = True

    # ------------------------------- Entry Position Update -------------------------------------
        if entry_flag:

            # Populate the trades dataframe
            trades = pd.DataFrame(index=[0])
            trades['entry_date'] = i
            trades['entry_price'] = round(data.loc[i, 'Close'], 2)
            trades['position'] = current_position

            stop_loss = data.loc[i, 'stoploss']

            take_profit = data.loc[i, 'target']

            # Increase the number of trades by 1
            trade_num += 1

            # Print trade details
            print(
                f"\033[34mTrade No: {trade_num}\033[0m | Entry Date: {i} | Entry Price: {trades.entry_price[0]} | Position: {current_position}")

            # Set exit flag to false
            entry_flag = False

            continue

    # ------------------------------- Exit Position Update -------------------------------------

        if exit_flag:

            # Populate the trades dataframe
            trades['exit_date'] = i
            trades['exit_type'] = exit_type
            trades['exit_price'] = round(data.loc[i, 'Close'], 2)

            # Calculate pnl for the trade
            trade_pnl = current_position * \
                (round(trades.exit_price[0] - trades.entry_price[0], 2))

            # Calculate cumulative pnl
            cum_pnl += trade_pnl
            cum_pnl = round(cum_pnl, 2)
            trades['PnL'] = trade_pnl

            # Add the trade logs to round trip details
            round_trips_details = pd.concat([round_trips_details, trades])

            # Print trade details
            print(
                f"Exit Type: {exit_type} | Exit Date: {i} | Exit Price: {trades.exit_price[0]} | PnL: {trade_pnl} | Cum PnL: {cum_pnl}")
            print("-"*30)

            # Update current position to 0
            current_position = 0

            # Set exit flag to false
            exit_flag = False

            continue
    return round_trips_details

Let's use this function to perform a backtest for all of the instances of the head and shoulders patterns. You can simply pass the data stored in `hs_strategy_data` to the `backtester` function and it will return the trade details. 

In [6]:
trade_details = backtester(hs_strategy_data)

Trade No: 1 | Entry Date: 1993-11-04 00:00:00 | Entry Price: 26.91 | Position: -1.0
Exit Type: SL | Exit Date: 1993-12-22 00:00:00 | Exit Price: 27.65 | PnL: -0.74 | Cum PnL: -0.74
------------------------------
Trade No: 2 | Entry Date: 2001-06-13 00:00:00 | Entry Price: 83.25 | Position: -1.0
Exit Type: TP | Exit Date: 2002-07-18 00:00:00 | Exit Price: 59.53 | PnL: 23.72 | Cum PnL: 22.98
------------------------------
Trade No: 3 | Entry Date: 2003-08-05 00:00:00 | Entry Price: 66.5 | Position: -1.0
Exit Type: SL | Exit Date: 2003-08-29 00:00:00 | Exit Price: 69.96 | PnL: -3.46 | Cum PnL: 19.52
------------------------------
Trade No: 4 | Entry Date: 2006-02-07 00:00:00 | Entry Price: 90.53 | Position: -1.0
Exit Type: SL | Exit Date: 2006-02-16 00:00:00 | Exit Price: 93.18 | PnL: -2.65 | Cum PnL: 16.87
------------------------------
Trade No: 5 | Entry Date: 2011-09-22 00:00:00 | Entry Price: 91.54 | Position: -1.0
Exit Type: SL | Exit Date: 2011-10-14 00:00:00 | Exit Price: 99.42 | 

As shown above, with the help of some print statements we have obtained a decent trade log which helps us understand the nature of each of the trades that were executed. 

For example, the very last short position was opened on 2018-10-05. And this short position was exited on 2018-10-10 as the take-profit (TP) level was hit. Hence, there is a profit of $8.89 associated with this particular trade.

Now let's display the first 5 rows of the `trade_details` that we have obtained by using the `backtester` function.

In [7]:
# Display the trade details
print(trade_details.tail())

  entry_date  entry_price  position  exit_date exit_type  exit_price    PnL
0 2006-02-07        90.53      -1.0 2006-02-16        SL       93.18  -2.65
0 2011-09-22        91.54      -1.0 2011-10-14        SL       99.42  -7.88
0 2015-08-20       178.67      -1.0 2015-11-03        SL      185.79  -7.12
0 2016-06-27       178.63      -1.0 2016-06-30        SL      187.47  -8.84
0 2018-10-05       268.94      -1.0 2018-10-24        TP      247.92  21.02


You will need the data present in the dataframe `trade_details` to further evaluate the performance of our strategy. For this, you can simply you can update the following cell's type to **Code** and run the code line, in order to save the data into a CSV file named `trades_hs_spy_1993_2018.csv`.

<a id='conclusion'></a>
## Conclusion and Next Steps
So far, you have learnt how to detect the formation of a head and shoulders pattern in the price data of any asset. Not only that, but, you have further also understood how to set the risk management parameters and backtest a complete strategy based on the head and shoulders pattern.

In the upcoming notebooks, we will perform a detailed analysis of the strategy's performance by looking at the trade-level analytics and also various performance metrics. So stay tuned! <br><br>